In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/weather-data/Dataset_for_Smart_irrigation.xlsx


In [3]:
import pandas as pd

# Replace 'your_file.xlsx' with the path to your Excel file
df = pd.read_excel('/kaggle/input/weather-data/Dataset_for_Smart_irrigation.xlsx')

# Display the first few rows
print(df.head())


          State_Name  Rainfall_value  Tmax_value  Tmin_value      Month  \
0  Andaman & Nicobar             NaN         NaN         NaN 2015-01-01   
1  Andaman & Nicobar             NaN         NaN         NaN 2015-02-01   
2  Andaman & Nicobar             NaN         NaN         NaN 2015-03-01   
3  Andaman & Nicobar             NaN         NaN         NaN 2015-04-01   
4  Andaman & Nicobar             NaN         NaN         NaN 2015-05-01   

        State_Name.1  Soil Moisture  Month.1  
0                NaN            NaN      NaN  
1                NaN            NaN      NaN  
2  Andaman & Nicobar       0.165186  2015-03  
3  Andaman & Nicobar       0.256472  2015-04  
4  Andaman & Nicobar       0.319297  2015-05  


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4440 entries, 0 to 4439
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   State_Name      4440 non-null   object        
 1   Rainfall_value  4200 non-null   float64       
 2   Tmax_value      4080 non-null   float64       
 3   Tmin_value      4080 non-null   float64       
 4   Month           4440 non-null   datetime64[ns]
 5   State_Name.1    4329 non-null   object        
 6   Soil Moisture   4329 non-null   float64       
 7   Month.1         4329 non-null   object        
dtypes: datetime64[ns](1), float64(4), object(3)
memory usage: 277.6+ KB


In [5]:
df.describe()

,Rainfall_value,Tmax_value,Tmin_value,Month,Soil Moisture
count,4200.000000,4080.000000,4080.000000,4440,4329.000000
mean,128.447312,30.367305,18.958199,2019-12-16 10:48:00,0.216899
min,0.000000,7.669185,-1.308141,2015-01-01 00:00:00,0.041844
25%,9.110497,28.342923,14.923933,2017-06-23 12:00:00,0.128713
50%,52.567013,30.920469,20.852983,2019-12-16 12:00:00,0.209657
75%,187.365028,33.354405,23.898877,2022-06-08 12:00:00,0.296231
max,2515.446045,42.563156,28.346670,2024-12-01 00:00:00,0.480837
std,189.724989,5.179561,6.105374,NaN,0.099006


In [6]:
# Check for nulls
print(df.isnull().sum())

State_Name          0
Rainfall_value    240
Tmax_value        360
Tmin_value        360
Month               0
State_Name.1      111
Soil Moisture     111
Month.1           111
dtype: int64


In [7]:
# After imputation, verify missing values
print(df[['Rainfall_value','Tmax_value','Tmin_value']].isnull().sum())


Rainfall_value    240
Tmax_value        360
Tmin_value        360
dtype: int64


In [8]:
# Ensure your Month column is datetime type
df['Month'] = pd.to_datetime(df['Month'])

# Set Month as the index
df = df.set_index('Month')


In [9]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def kalman_impute(series, order=(1,1,1), seasonal_order=(1,1,1,12)):
    try:
        model = SARIMAX(series, order=order, seasonal_order=seasonal_order)
        results = model.fit(disp=False)
        imputed = results.predict(start=series.index, end=series.index[-1])
        # Fill only the missing values
        series_filled = series.copy()
        series_filled[series.isnull()] = imputed[series.isnull()]
        return series_filled
    except:
        # Fallback to time-based interpolation
        return series.interpolate(method='time')


In [10]:
impute_cols = ['Rainfall_value', 'Tmax_value', 'Tmin_value']
for col in impute_cols:
    df[col] = kalman_impute(df[col])


/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it is not monotonic and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it is not monotonic and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsm

In [11]:
# After imputation, verify missing values
print(df[['Rainfall_value','Tmax_value','Tmin_value']].isnull().sum())


Rainfall_value    120
Tmax_value        120
Tmin_value        120
dtype: int64


In [12]:
df = df.drop('State_Name.1', axis=1)


In [13]:
df.head(5)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,State_Name,Rainfall_value,Tmax_value,Tmin_value,Soil Moisture,Month.1
Month,,,,,,
2015-01-01,Andaman & Nicobar,NaN,NaN,NaN,NaN,NaN
2015-02-01,Andaman & Nicobar,NaN,NaN,NaN,NaN,NaN
2015-03-01,Andaman & Nicobar,NaN,NaN,NaN,0.165186,2015-03
2015-04-01,Andaman & Nicobar,NaN,NaN,NaN,0.256472,2015-04
2015-05-01,Andaman & Nicobar,NaN,NaN,NaN,0.319297,2015-05


In [14]:
df = df.drop('Month.1', axis=1)


In [15]:
df.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,State_Name,Rainfall_value,Tmax_value,Tmin_value,Soil Moisture
Month,,,,,
2015-01-01,Andaman & Nicobar,NaN,NaN,NaN,NaN
2015-02-01,Andaman & Nicobar,NaN,NaN,NaN,NaN
2015-03-01,Andaman & Nicobar,NaN,NaN,NaN,0.165186
2015-04-01,Andaman & Nicobar,NaN,NaN,NaN,0.256472
2015-05-01,Andaman & Nicobar,NaN,NaN,NaN,0.319297


In [16]:
df.sample(5)

,State_Name,Rainfall_value,Tmax_value,Tmin_value,Soil Moisture
Month,,,,,
2023-06-01,Himachal Pradesh,110.356094,31.259256,19.776716,0.246803
2023-11-01,Puducherry,14.039487,38.160641,27.886509,0.104489
2021-04-01,Daman and Diu and Dadra and Nagar Haveli,0.512578,36.673035,23.513933,0.119370
2019-01-01,Puducherry,73.653175,30.764984,22.874670,0.171234
2023-08-01,Telengana,89.046707,32.289497,24.027916,0.353928


In [17]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def kalman_impute(series, order=(1,1,1), seasonal_order=(1,1,1,12)):
    try:
        model = SARIMAX(series, order=order, seasonal_order=seasonal_order)
        results = model.fit(disp=False)
        imputed = results.predict(start=series.index[0], end=series.index[-1])
        series_filled = series.copy()
        series_filled[series.isnull()] = imputed[series.isnull()]
        return series_filled
    except Exception as e:
        print(f"Kalman failed: {e}, using linear interpolation.")
        return series.interpolate(method='time')

# Apply state-wise
for state in df['State_Name'].unique():
    mask = df['State_Name'] == state
    series = df.loc[mask, 'Soil Moisture']
    df.loc[mask, 'Soil Moisture'] = kalman_impute(series)


/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used

Kalman failed: boolean index did not match indexed array along dimension 0; dimension is 120 but corresponding boolean dimension is 240, using linear interpolation.


/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferr

In [18]:
# After imputation, verify missing values
print(df[['Rainfall_value','Tmax_value','Tmin_value','State_Name','Soil Moisture']].isnull().sum())


Rainfall_value    120
Tmax_value        120
Tmin_value        120
State_Name          0
Soil Moisture       2
dtype: int64


In [19]:
# First, try forward fill
df['Soil Moisture'] = df['Soil Moisture'].fillna(method='ffill')

/tmp/ipykernel_35/657184539.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Soil Moisture'] = df['Soil Moisture'].fillna(method='ffill')


In [20]:
# After imputation, verify missing values
print(df[['Rainfall_value','Tmax_value','Tmin_value','State_Name','Soil Moisture']].isnull().sum())


Rainfall_value    120
Tmax_value        120
Tmin_value        120
State_Name          0
Soil Moisture       0
dtype: int64


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4440 entries, 2015-01-01 to 2024-12-01
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   State_Name      4440 non-null   object 
 1   Rainfall_value  4320 non-null   float64
 2   Tmax_value      4320 non-null   float64
 3   Tmin_value      4320 non-null   float64
 4   Soil Moisture   4440 non-null   float64
dtypes: float64(4), object(1)
memory usage: 208.1+ KB


In [22]:
df = pd.get_dummies(df, columns=['State_Name'], dtype=int)


In [23]:
df.sample()

,Rainfall_value,Tmax_value,Tmin_value,Soil Moisture,State_Name_Andaman & Nicobar,State_Name_Andhra Pradesh,State_Name_Arunachal Pradesh,State_Name_Assam,State_Name_Bihar,State_Name_Chandigarh,...,State_Name_Puducherry,State_Name_Punjab,State_Name_Rajasthan,State_Name_Sikkim,State_Name_Tamilnadu,State_Name_Telengana,State_Name_Tripura,State_Name_Uttar Pradesh,State_Name_Uttarakhand,State_Name_West Bengal
Month,,,,,,,,,,,,,,,,,,,,,
2022-05-01,61.461754,32.451847,18.901253,0.217671,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
df = df.sort_index()


In [25]:
# Check time differences between index entries
print(df.index.to_series().diff().value_counts())


Month
0 days     4320
31 days      69
30 days      40
28 days       7
29 days       3
Name: count, dtype: int64


In [26]:
# Set thresholds (adjust as needed for your crop/region)
SOIL_MOISTURE_THRESHOLD = 30  # percent volumetric
RAINFALL_THRESHOLD = 50       # mm per month

# Create label
df['Irrigate'] = ((df['Soil Moisture'] < SOIL_MOISTURE_THRESHOLD) & 
                  (df['Rainfall_value'] < RAINFALL_THRESHOLD)).astype(int)


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)


In [27]:
df.sample(5)

,Rainfall_value,Tmax_value,Tmin_value,Soil Moisture,State_Name_Andaman & Nicobar,State_Name_Andhra Pradesh,State_Name_Arunachal Pradesh,State_Name_Assam,State_Name_Bihar,State_Name_Chandigarh,...,State_Name_Punjab,State_Name_Rajasthan,State_Name_Sikkim,State_Name_Tamilnadu,State_Name_Telengana,State_Name_Tripura,State_Name_Uttar Pradesh,State_Name_Uttarakhand,State_Name_West Bengal,Irrigate
Month,,,,,,,,,,,,,,,,,,,,,
2019-06-01,543.963257,30.980240,23.123392,0.285027,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2023-10-01,92.513428,32.472435,23.436432,0.206058,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2018-10-01,73.338921,31.262590,20.019068,0.342214,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2024-11-01,29.042095,27.540407,15.443287,0.199115,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2020-02-01,27.780447,21.886057,9.908293,0.208433,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,1


In [28]:
df['Irrigate'].value_counts()


Irrigate
0    2314
1    2126
Name: count, dtype: int64

In [29]:
# 1. Sort by date to maintain temporal order
if 'Month' in df.columns:
    df = df.sort_values('Month')
elif df.index.name != 'Month':
    df = df.sort_index()

# 2. Define train-test split ratio (e.g., 80% train, 20% test)
train_ratio = 0.8
train_size = int(len(df) * train_ratio)

# 3. Split data chronologically
train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

# 4. (Optional) Check label distribution in train and test sets
print("Train label counts:\n", train_df['Irrigate'].value_counts())
print("Test label counts:\n", test_df['Irrigate'].value_counts())


Train label counts:
 Irrigate
0    1848
1    1704
Name: count, dtype: int64
Test label counts:
 Irrigate
0    466
1    422
Name: count, dtype: int64


In [30]:
from sklearn.preprocessing import StandardScaler

# List your continuous feature columns
feature_cols = ['Rainfall_value', 'Tmax_value', 'Tmin_value', 'Soil Moisture']

# Initialize the scaler
scaler = StandardScaler()

# Fit scaler on the training set only
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])

# Transform the test set using the same scaler
test_df[feature_cols] = scaler.transform(test_df[feature_cols])


In [39]:
def create_sequences(features, labels, window_size):
    X, y = [], []
    for i in range(len(features) - window_size):
        X.append(features[i:i+window_size])
        y.append(labels[i+window_size])
    return np.array(X), np.array(y)

window_size = 12  # e.g., use last 6 months to predict next
feature_cols = ['Rainfall_value', 'Tmax_value', 'Tmin_value', 'Soil Moisture'] + \
               [col for col in train_df.columns if col.startswith('State_Name_')]

X_train, y_train = create_sequences(train_df[feature_cols].values, train_df['Irrigate'].values, window_size)
X_test, y_test = create_sequences(test_df[feature_cols].values, test_df['Irrigate'].values, window_size)


In [44]:
from tensorflow import keras
from tensorflow.keras import layers, Model

input_shape = X_train.shape[1:]  # (window_size, num_features)

# Input layer
input_layer = layers.Input(shape=input_shape)

# TCN Branch
tcn = layers.Conv1D(filters=64, kernel_size=3, dilation_rate=1, padding='causal', activation='relu')(input_layer)
tcn = layers.Conv1D(filters=64, kernel_size=3, dilation_rate=2, padding='causal', activation='relu')(tcn)
tcn = layers.Conv1D(filters=64, kernel_size=3, dilation_rate=4, padding='causal', activation='relu')(tcn)
tcn = layers.GlobalAveragePooling1D()(tcn)  # Shape: (None, 64)

# LSTM Branch
lstm = layers.LSTM(64, return_sequences=True)(input_layer)
lstm = layers.LSTM(32)(lstm)  # Shape: (None, 32)

# Combine branches
combined = layers.Concatenate()([tcn, lstm])  # Shape: (None, 96)

# Attention Mechanism (Feature-wise)
attention_weights = layers.Dense(96, activation='sigmoid')(combined)  # Shape: (None, 96)
context = layers.Multiply()([attention_weights, combined])  # Shape: (None, 96)

# Output layers
output = layers.Dense(64, activation='relu')(context)
output = layers.Dense(1, activation='sigmoid')(output)

# Build model
model = Model(inputs=input_layer, outputs=output)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4             │ (None, 12, 40)         │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_11 (Conv1D)        │ (None, 12, 64)         │          7,744 │ input_layer_4[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_12 (Conv1D)        │ (None, 12, 64)         │         12,352 │ conv1d_11[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_13 (Conv1D)        │ (None, 12, 64)         │         12,352 │ conv1d_12[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_2 (LSTM)             │ (None, 12, 64)         │         26,880 │ input_layer_4[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ global_average_pooling1d… │ (None, 64)             │              0 │ conv1d_13[0][0]        │
│ (GlobalAveragePooling1D)  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_3 (LSTM)             │ (None, 32)             │         12,416 │ lstm_2[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_1             │ (None, 96)             │              0 │ global_average_poolin… │
│ (Concatenate)             │                        │                │ lstm_3[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_8 (Dense)           │ (None, 96)             │          9,312 │ concatenate_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multiply (Multiply)       │ (None, 96)             │              0 │ dense_8[0][0],         │
│                           │                        │                │ concatenate_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_9 (Dense)           │ (None, 64)             │          6,208 │ multiply[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_10 (Dense)          │ (None, 1)              │             65 │ dense_9[0][0]          │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 87,329 (341.13 KB)

 Trainable params: 87,329 (341.13 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test)
)


Epoch 1/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.5178 - loss: nan - val_accuracy: 0.5320 - val_loss: nan
Epoch 2/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5266 - loss: nan - val_accuracy: 0.5320 - val_loss: nan
Epoch 3/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5173 - loss: nan - val_accuracy: 0.5320 - val_loss: nan
Epoch 4/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5215 - loss: nan - val_accuracy: 0.5320 - val_loss: nan
Epoch 5/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5267 - loss: nan - val_accuracy: 0.5320 - val_loss: nan
Epoch 6/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5210 - loss: nan - val_accuracy: 0.5320 - val_loss: nan
Epoch 7/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5387 - loss: nan - val_accuracy: 0.5320 - val_loss: nan
Epoch 8/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5193 - loss: nan - val_accuracy: 0.5320 - val_loss: nan
Epoch 9/30
111/111 ━━━━

In [46]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.2f}")


28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4793 - loss: nan
Test Accuracy: 0.53
